<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/Normalisation/03-lab-normalisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 3 Lab — Normalisation: BCNF Design and Star Schema Comparison

This lab has three parts. Part 1 diagnoses normal form violations on standalone tables (pen-and-paper reasoning). Part 2 builds the Orders & Payments BCNF schema on dbdiagram.io and adds a justified downshift. Part 3 designs a star schema, loads pre-built data into MySQL, and compares a reporting query against both schemas.

Use the formal vocabulary (functional dependency, candidate key, determinant, normal form, decomposition, star schema, fact table, dimension table, grain) and reference specific chapter tables (Tables 1–8) throughout your work.

**Time budget:** Part 1 ~30 min | Part 2 ~40 min | Part 3 ~35 min | **Total ~105 min**

**Tools required:**
- [dbdiagram.io](https://dbdiagram.io/) — free, browser-based, DDL export (Parts 2 and 3)
- Google Colab with MySQL — for SQL execution (Part 3)

---

## Part 1 — Identify Normal Form (Pen-and-Paper Diagnosis)

These four tables are independent of the Orders & Payments domain. For each, you are given the table schema and its functional dependencies. Your task is to:

1. State the current normal form.
2. Identify the violating FD (if any).
3. Propose the decomposition that fixes it.

Work through each table before checking the solutions. This is the *recognition* skill from Algorithm 5.2 in the chapter — naming the normal form, not building to it.

---

### Q1. Not 1NF — Student Course Registration

A university stores student registrations in a single table:

```
StudentRecord(student_id, student_name, courses)
```

Sample data:

| student_id | student_name | courses |
|---|---|---|
| S001 | Alice Chen | Math, Physics, Chemistry |
| S002 | Bob Patel | Math, Biology |
| S003 | Carol Liu | Physics |

**(a)** Is this table in 1NF? Justify your answer by checking the 1NF rule from Table 3 in the chapter.

**(b)** What specific problem does the `courses` column create for querying? Consider: how would you write a query to find all students enrolled in Physics?

**(c)** Propose a decomposition that brings this table to 1NF. Draw the resulting table(s) and show what the sample data looks like after decomposition.

**(d)** After your decomposition, what is the primary key of each resulting table?

---

### Q2. 1NF but not 2NF — Enrolment with Partial Dependency

A university enrolment table with a composite key:

```
Enrolment(student_id, course_id, student_name, course_title, grade)
```

**Functional dependencies:**
- `{student_id, course_id} → grade` (the full composite key determines the grade)
- `student_id → student_name` (name depends on student alone)
- `course_id → course_title` (title depends on course alone)

**(a)** *Predict before analysing:* Is this table in 2NF? Use the 2NF check from Table 6 in the chapter: "Does every non-key attribute depend on the *whole* key?"

**(b)** Identify the violating FDs. Which non-key attributes depend on only *part* of the composite key?

**(c)** What concrete redundancy does this cause? If student S001 is enrolled in three courses, how many times is their name stored?

**(d)** Propose the decomposition to 2NF. How many tables result? State the primary key and columns of each.

**(e)** After your decomposition, check: is each resulting table also in 3NF and BCNF? Justify.

---

### Q3. 2NF but not 3NF — Order with Transitive Dependency

An order table where customer details are stored alongside the order:

```
OrderFlat(order_id, customer_id, customer_name, customer_city, order_date, total)
```

**Functional dependencies:**
- `order_id → customer_id, customer_name, customer_city, order_date, total`
- `customer_id → customer_name, customer_city`

**(a)** *Predict before analysing:* This table has a single-column primary key (`order_id`). Can it have a 2NF violation? Use the 2NF check from Table 6 in the chapter to justify your answer.

**(b)** Identify the transitive dependency. Which non-key attribute depends on another non-key attribute?

**(c)** This is the same structural problem as Failure 1 in the chapter. Explain concretely: if you update `customer_city` for customer C001 in the `Customer` record but not in existing orders, what happens to a "revenue by city" report?

**(d)** Propose the decomposition to 3NF. State each resulting table with its columns and primary key.

**(e)** *Checkpoint:* After the decomposition, retrieving a customer's city requires a JOIN. Is this extra complexity a bug or a feature? Connect your answer to the chapter's central trade-off.

---

### Q4. BCNF — A Well-Designed Table

The `Payment` table from the Orders & Payments domain:

```
Payment(payment_id, order_id, amount, payment_date, method)
```

**Functional dependencies:**
- `payment_id → order_id, amount, payment_date, method`
- `order_id → payment_id, amount, payment_date, method` (because `order_id` is UNIQUE — one payment per order)

**Constraints:** `order_id` has a UNIQUE constraint.

**(a)** List all candidate keys. (Hint: a candidate key is a minimal set of attributes that determines all others.)

**(b)** For each FD, check: is the determinant a candidate key?

**(c)** State the normal form. Confirm that no BCNF violation exists.

**(d)** *Checkpoint:* If ShopEasy later decides to support instalment payments (multiple payments per order), what changes? Does `order_id` remain a candidate key? How does this affect the normal form analysis?

---

## Part 2 — BCNF Design on dbdiagram.io (OLTP Truth)

### Setup

Open [dbdiagram.io](https://dbdiagram.io/) and start a new diagram. You will build the Orders & Payments BCNF schema here, extending the Chapter 2 schema with normalisation refinements.

---

### Q5. Build the BCNF Core Schema

Starting from the Chapter 2 schema (Section 6.1 of the chapter), build the BCNF core on dbdiagram.io. Include these tables:

- `Postcode(postcode PK, city)` — new lookup table, decomposed from Customer
- `Customer(customer_id PK, name, email, postcode FK)`
- `Order_(order_id PK, customer_id FK, order_date, status)`
- `Product(product_id PK, name, price, quantity_in_stock)`
- `OrderLine(order_id FK, product_id FK, quantity, unit_price)` — composite PK
- `Payment(payment_id PK, order_id FK UNIQUE, amount, payment_date, method)`
- `Shipping(shipping_id PK, order_id FK UNIQUE, address, shipped_date, status)`
- `Review(review_id PK, customer_id FK, product_id FK, rating, comment, review_date)`
- `Supplier(supplier_id PK, name, contact_email)`
- `SupplierProduct(supplier_id FK, product_id FK)` — composite PK
- `Category(category_id PK, name, description)`
- `ProductCategory(product_id FK, category_id FK)` — composite PK

**(a)** What is new compared to the Chapter 2 schema? Identify the table that was added and the column that was changed.

**(b)** *Predict before building:* How many tables does this schema have? Count, then verify in dbdiagram.io.

---

### Q6. Verify BCNF for Each Table

For each of the following tables, list the functional dependencies, identify candidate keys, and confirm that every determinant is a candidate key.

**(a)** `Customer(customer_id, name, email, postcode)` — Is `email` also a candidate key?

**(b)** `OrderLine(order_id, product_id, quantity, unit_price)` — Are there any partial dependencies?

**(c)** `Payment(payment_id, order_id, amount, payment_date, method)` — How many candidate keys does this table have?

**(d)** For any table where you find a violation, propose the decomposition. (You should find none — the schema was designed to BCNF.)

---

### Q7. Add a Justified Downshift

Choose **one** of the following downshifts and add it to your dbdiagram.io schema:

**Option A — Snapshot:** Add `shipping_address`, `shipping_city`, and `shipping_postcode` columns to the `Shipping` table. These record the delivery address at time of shipment.

**Option B — Derived attribute:** Add an `order_total` column to the `Order_` table. This holds the sum of `(quantity × unit_price)` from the order's `OrderLine` rows.

**(a)** State which downshift you chose and add it to your dbdiagram.io schema.

**(b)** Name the downshift type (snapshot or derived attribute).

**(c)** State which normal form is being relaxed. Is it a *real* NF violation or an apparent one? (Hint: review the chapter's distinction between snapshots and copies.)

**(d)** State the safety mechanism: how is consistency maintained?

**(e)** Add a comment to your dbdiagram.io schema documenting the downshift. The comment should include: the type, the justification, and the safety mechanism.

---

## Part 3 — Star Schema: Design, Load, and Compare

### Setup

Run the following cells in a new Google Colab notebook. The setup creates both schemas (BCNF and star), loads seed data using pure SQL, and then derives the star schema from the BCNF source — demonstrating the "truth is normalised; reporting is derived" principle in action.

**Cell 1 — Install and connect:**

In [ ]:
# ── Install and start MySQL server on Colab ─────────────────
!apt-get update -qq && apt-get install -y -qq mysql-server > /dev/null 2>&1
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';" 2>/dev/null || true
!mysql -e "SELECT 'MySQL is ready!' AS status;"


In [ ]:
# Topic 3 — Normalisation: BCNF vs Star Schema Comparison
!pip install -q ipython-sql==0.5.0
!pip install -q pymysql==1.1.0
!pip install -q sqlalchemy==2.0.20
!pip install prettytable==2.0.0

# Load SQL magic
%load_ext sql

# Connect to MySQL (adjust credentials for your Colab environment)
%sql mysql+pymysql://root:@localhost/shopdb


**Cell 2 — Create the BCNF schema:**

In [ ]:
%%sql
DROP DATABASE IF EXISTS shopdb_bcnf;
CREATE DATABASE shopdb_bcnf;
USE shopdb_bcnf;

-- Postcode lookup (BCNF decomposition from Chapter 3)
CREATE TABLE Postcode (
    postcode VARCHAR(10) PRIMARY KEY,
    city     VARCHAR(50) NOT NULL
);

CREATE TABLE Customer (
    customer_id INT PRIMARY KEY,
    name        VARCHAR(100) NOT NULL,
    email       VARCHAR(100) NOT NULL UNIQUE,
    postcode    VARCHAR(10),
    FOREIGN KEY (postcode) REFERENCES Postcode(postcode)
);

CREATE TABLE Department (
    department_id   INT PRIMARY KEY,
    department_name VARCHAR(50) NOT NULL
);

CREATE TABLE Category (
    category_id   INT PRIMARY KEY,
    category_name VARCHAR(50) NOT NULL,
    department_id INT NOT NULL,
    FOREIGN KEY (department_id) REFERENCES Department(department_id)
);

CREATE TABLE Product (
    product_id       INT PRIMARY KEY,
    product_name     VARCHAR(100) NOT NULL,
    price            DECIMAL(10,2) NOT NULL,
    category_id      INT NOT NULL,
    quantity_in_stock INT NOT NULL DEFAULT 0,
    FOREIGN KEY (category_id) REFERENCES Category(category_id)
);

CREATE TABLE Supplier (
    supplier_id   INT PRIMARY KEY,
    supplier_name VARCHAR(100) NOT NULL,
    region        VARCHAR(50)
);

CREATE TABLE Order_ (
    order_id    INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date  DATE NOT NULL,
    status      VARCHAR(20) DEFAULT 'completed',
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id)
);

CREATE TABLE OrderLine (
    order_id   INT NOT NULL,
    product_id INT NOT NULL,
    quantity   INT NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES Order_(order_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id)
);


**Cell 3 — Load BCNF seed data:**

In [ ]:
%%sql
USE shopdb_bcnf;

-- ── Reference data ──────────────────────────────────────────
INSERT INTO Postcode VALUES
  ('018956','Singapore'),('049315','Singapore'),('50088','Kuala Lumpur'),
  ('50450','Kuala Lumpur'),('10110','Bangkok'),('10500','Bangkok'),
  ('12190','Jakarta'),('10310','Jakarta'),('1000','Manila'),
  ('700000','Ho Chi Minh City');

INSERT INTO Department VALUES
  (1,'Electronics'),(2,'Clothing'),(3,'Home & Garden'),(4,'Books'),(5,'Sports');

INSERT INTO Category VALUES
  (1,'Laptops',1),(2,'Phones',1),(3,'Accessories',1),
  (4,'Tops',2),(5,'Trousers',2),(6,'Footwear',2),
  (7,'Furniture',3),(8,'Kitchen',3),(9,'Garden Tools',3),
  (10,'Fiction',4),(11,'Non-Fiction',4),(12,'Technical',4),
  (13,'Equipment',5),(14,'Apparel',5),(15,'Nutrition',5);

INSERT INTO Product VALUES
  (1,'Budget Laptop',349.99,1,80),(2,'Pro Laptop',899.99,1,35),
  (3,'Basic Phone',149.99,2,120),(4,'Flagship Phone',699.99,2,50),
  (5,'USB Cable',9.99,3,500),(6,'Phone Case',19.99,3,300),
  (7,'Cotton Tee',24.99,4,200),(8,'Silk Blouse',59.99,4,75),
  (9,'Jeans',49.99,5,150),(10,'Chinos',39.99,5,100),
  (11,'Trainers',79.99,6,90),(12,'Boots',99.99,6,60),
  (13,'Sofa',499.99,7,20),(14,'Bookshelf',129.99,7,45),
  (15,'Toaster',34.99,8,110),(16,'Blender',44.99,8,85),
  (17,'Lawn Mower',199.99,9,30),(18,'Pruning Shears',14.99,9,200),
  (19,'Mystery Novel',12.99,10,300),(20,'Sci-Fi Epic',14.99,10,250),
  (21,'History Book',18.99,11,180),(22,'Biography',16.99,11,150),
  (23,'SQL Handbook',39.99,12,100),(24,'Data Modelling Guide',44.99,12,80),
  (25,'Tennis Racket',89.99,13,55),(26,'Football',29.99,13,120),
  (27,'Running Shorts',34.99,14,140),(28,'Gym Vest',19.99,14,200),
  (29,'Protein Powder',24.99,15,250),(30,'Energy Bars',9.99,15,400);

INSERT INTO Supplier VALUES
  (1,'TechCorp','Singapore'),(2,'FashionHub','Malaysia'),
  (3,'HomeBase','Thailand'),(4,'BookWorld','Indonesia'),(5,'SportsPro','Philippines');


**Cell 4 — Load customers and orders (procedural generation):**

In [ ]:
# Generate customers and orders using pure SQL via a helper
# This avoids row-by-row Python inserts for speed and reliability
import random
from datetime import date, timedelta

random.seed(42)

cities_map = {
    '018956':'Singapore','049315':'Singapore','50088':'Kuala Lumpur',
    '50450':'Kuala Lumpur','10110':'Bangkok','10500':'Bangkok',
    '12190':'Jakarta','10310':'Jakarta','1000':'Manila',
    '700000':'Ho Chi Minh City'
}
postcodes = list(cities_map.keys())

# ── Build bulk INSERT statements for speed ──
# Customers (200)
cust_rows = []
for i in range(1, 201):
    pc = random.choice(postcodes)
    cust_rows.append(f"({i},'Customer {i}','customer{i}@example.com','{pc}')")
cust_sql = "INSERT INTO Customer VALUES\n  " + ",\n  ".join(cust_rows) + ";"

# Orders (12,000) and OrderLines (~50,000)
start_date = date(2023, 1, 1)
order_rows = []
ol_rows = []
for oid in range(1, 12001):
    cid = random.randint(1, 200)
    odate = start_date + timedelta(days=random.randint(0, 729))
    order_rows.append(f"({oid},{cid},'{odate}','completed')")
    # 1-6 products per order (from 30 products), no duplicates
    n_items = random.randint(1, 6)
    prods = random.sample(range(1, 31), n_items)
    for pid in prods:
        qty = random.randint(1, 10)
        up = round(random.uniform(5.0, 500.0), 2)
        ol_rows.append(f"({oid},{pid},{qty},{up})")

# Execute in batches of 2000 for reliability
def batch_insert(table, rows, batch_size=2000):
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        sql = f"INSERT INTO {table} VALUES\n  " + ",\n  ".join(batch) + ";"
        %sql $sql

%sql USE shopdb_bcnf;
batch_insert("Customer", cust_rows)
batch_insert("Order_", order_rows)
batch_insert("OrderLine", ol_rows)

print(f"BCNF schema loaded: 200 customers, {len(order_rows)} orders, {len(ol_rows)} order lines")


**Cell 5 — Create and populate the star schema (derived from BCNF source):**

This is the key step: the star schema is built *from* the normalised BCNF source using SQL queries. This is the "truth is normalised; reporting is derived" principle in action.

In [ ]:
%%sql
DROP DATABASE IF EXISTS shopdb_star;
CREATE DATABASE shopdb_star;
USE shopdb_star;

-- ── Dimension tables ────────────────────────────────────────

-- TimeDim: derived from distinct order dates
CREATE TABLE TimeDim (
    date_key  INT PRIMARY KEY AUTO_INCREMENT,
    full_date DATE NOT NULL UNIQUE,
    month     INT NOT NULL,
    quarter   INT NOT NULL,
    year      INT NOT NULL
);

INSERT INTO TimeDim (full_date, month, quarter, year)
SELECT DISTINCT
    o.order_date,
    MONTH(o.order_date),
    QUARTER(o.order_date),
    YEAR(o.order_date)
FROM shopdb_bcnf.Order_ o
ORDER BY o.order_date;

-- ProductDim: derived from Product + Category + Department (flattened)
-- NOTE: category_name → department_name is a DELIBERATE 3NF violation
CREATE TABLE ProductDim (
    product_key     INT PRIMARY KEY,
    product_name    VARCHAR(100) NOT NULL,
    category_name   VARCHAR(50) NOT NULL,
    department_name VARCHAR(50) NOT NULL
);

INSERT INTO ProductDim
SELECT p.product_id, p.product_name, c.category_name, d.department_name
FROM shopdb_bcnf.Product p
  JOIN shopdb_bcnf.Category c   ON p.category_id = c.category_id
  JOIN shopdb_bcnf.Department d ON c.department_id = d.department_id;

-- CustomerDim: derived from Customer + Postcode (flattened)
-- NOTE: city → region would be a 3NF violation if region existed in BCNF;
--       here we map city to region for reporting convenience
CREATE TABLE CustomerDim (
    customer_key  INT PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    city          VARCHAR(50),
    region        VARCHAR(50)
);

INSERT INTO CustomerDim
SELECT cu.customer_id, cu.name, pc.city,
    CASE pc.city
        WHEN 'Singapore'      THEN 'Singapore'
        WHEN 'Kuala Lumpur'   THEN 'Malaysia'
        WHEN 'Bangkok'        THEN 'Thailand'
        WHEN 'Jakarta'        THEN 'Indonesia'
        WHEN 'Manila'         THEN 'Philippines'
        WHEN 'Ho Chi Minh City' THEN 'Vietnam'
    END AS region
FROM shopdb_bcnf.Customer cu
  LEFT JOIN shopdb_bcnf.Postcode pc ON cu.postcode = pc.postcode;

-- ── Fact table ──────────────────────────────────────────────
-- Grain: one row = one line item in one order
CREATE TABLE OrderLineFact (
    order_line_id INT PRIMARY KEY AUTO_INCREMENT,
    product_key  INT NOT NULL,
    date_key     INT NOT NULL,
    customer_key INT NOT NULL,
    quantity     INT NOT NULL,
    line_total   DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (product_key)  REFERENCES ProductDim(product_key),
    FOREIGN KEY (date_key)     REFERENCES TimeDim(date_key),
    FOREIGN KEY (customer_key) REFERENCES CustomerDim(customer_key)
);

INSERT INTO OrderLineFact (product_key, date_key, customer_key, quantity, line_total)
SELECT
    ol.product_id,
    td.date_key,
    o.customer_id,
    ol.quantity,
    ROUND(ol.quantity * ol.unit_price, 2)
FROM shopdb_bcnf.OrderLine ol
  JOIN shopdb_bcnf.Order_ o ON ol.order_id = o.order_id
  JOIN shopdb_star.TimeDim td ON o.order_date = td.full_date;


**Cell 6 — Verify setup:**

In [ ]:
%%sql
SELECT 'BCNF OrderLines' AS source, COUNT(*) AS row_count FROM shopdb_bcnf.OrderLine
UNION ALL
SELECT 'Star Facts', COUNT(*) FROM shopdb_star.OrderLineFact;
-- Expected: both counts should match (approximately 50,000)


---

### Q8. Design the Star Schema on dbdiagram.io

Before running any queries, design the star schema on dbdiagram.io to understand its structure.

**(a)** Create the star schema with these tables:
- `OrderLineFact(order_line_id PK, product_key FK, date_key FK, customer_key FK, quantity, line_total)`
- `ProductDim(product_key PK, product_name, category_name, department_name)`
- `TimeDim(date_key PK, full_date, month, quarter, year)`
- `CustomerDim(customer_key PK, customer_name, city, region)`

**(b)** State the **grain** of the fact table in one sentence: "One row represents ___."

**(c)** *Predict before analysing:* Which dimension tables do you expect to have 3NF violations? (Hint: look for columns that might have transitive dependencies — where one non-key column determines another.) Now work through the FD analysis for each dimension table: identify any FD where the determinant is not a candidate key, and state the normal form.

**(d)** For each 3NF violation you found in (c), explain why it is acceptable in the star schema context. Reference the three safety conditions from Section 4.4 of the chapter.

---

### Q9. Run the Reporting Query — BCNF Version

The reporting question: **"What is the total revenue by department by quarter?"**

**(a)** *Predict before running:* How many JOIN clauses does this query need against the BCNF schema? List the tables that must be joined.

Now run the query:

In [ ]:
%%sql
USE shopdb_bcnf;

SELECT d.department_name,
       QUARTER(o.order_date) AS q,
       YEAR(o.order_date)    AS y,
       SUM(ol.quantity * ol.unit_price) AS revenue
FROM OrderLine ol
  JOIN Product p   ON ol.product_id = p.product_id
  JOIN Category c  ON p.category_id = c.category_id
  JOIN Department d ON c.department_id = d.department_id
  JOIN Order_ o    ON ol.order_id = o.order_id
GROUP BY d.department_name, YEAR(o.order_date), QUARTER(o.order_date)
ORDER BY y, q, d.department_name;


**(b)** Record the execution time shown by the `%%sql` magic.

**(c)** Count the number of JOIN clauses. How many tables does this query touch?

**(d)** Examine the result: does it make sense? Are all five departments represented? Do the quarterly totals look reasonable given ~50,000 order lines?

---

### Q10. Run the Reporting Query — Star Schema Version

**(a)** *Predict before running:* How many JOIN clauses does this query need against the star schema? Which tables are joined?

Now run the query:

In [ ]:
%%sql
USE shopdb_star;

SELECT pd.department_name,
       td.quarter  AS q,
       td.year     AS y,
       SUM(f.line_total) AS revenue
FROM OrderLineFact f
  JOIN ProductDim pd ON f.product_key = pd.product_key
  JOIN TimeDim td    ON f.date_key = td.date_key
GROUP BY pd.department_name, td.year, td.quarter
ORDER BY y, q, pd.department_name;


**(b)** Record the execution time.

**(c)** Count the JOIN clauses. How many tables does this query touch?

**(d)** Compare the revenue totals to the BCNF version. Do they match? (Minor rounding differences are acceptable; structural differences are not.)

---

### Q11. Compare and Reflect

**(a)** Fill in the comparison table:

| Metric | BCNF query | Star schema query |
|---|---|---|
| Number of JOIN clauses | ___ | ___ |
| Number of tables touched | ___ | ___ |
| Execution time | ___ ms | ___ ms |
| Revenue totals match? | — | Yes / No |

**(b)** The star schema query has fewer JOINs but produces the same totals. Why? What structural property of the star schema makes this possible?

**(c)** What is the source of truth for `department_name` — the normalised `Department` table or the `ProductDim` dimension table? Justify your answer using the "truth is normalised; reporting is derived" principle from Section 4.4 of the chapter.

**(d)** *Checkpoint:* If a department is renamed in the OLTP system, what must happen to the star schema? What would go wrong if someone renamed it directly in `ProductDim`?

**(e)** **Note for Chapter 4:** At this point you have not learned JOIN syntax or the SQL execution pipeline — you ran these queries on faith to *feel* the performance difference. Chapter 4 will teach you how to read and trace multi-table queries. Chapter 4's worked examples will return to these exact queries and explain them using the vocabulary you will learn there. For now, record your observation: the structural difference (5 tables vs 3 tables, 4 JOINs vs 2 JOINs) is a direct consequence of the design choice (BCNF vs star schema).

---

### Q12. Bonus — Add a Drill-Down Filter

If time permits, add a WHERE clause to both queries to filter for a single department (e.g., `WHERE d.department_name = 'Electronics'` or `WHERE pd.department_name = 'Electronics'`).

**(a)** Run both filtered queries and record execution times.

**(b)** Does the performance gap widen, narrow, or stay roughly the same? Why might this be the case?

---

## Deliverable

- **Part 1** (Q1–Q4): For each of the four tables: current normal form, violating FD, and decomposition (or confirmation of BCNF compliance).
- **Part 2** (Q5–Q7): dbdiagram.io schema for the BCNF core with one documented downshift, plus BCNF verification notes for each table.
- **Part 3** (Q8–Q11): Star schema on dbdiagram.io, completed comparison table (Q11a), and written answers to the reflection questions.

The BCNF schema carries forward: Chapter 4 queries it with SQL, and Chapter 5 hardens it with CHECK constraints, CASCADE behaviours, views, and transactions.

---

## Solutions

Complete your own work before reviewing the solutions below. Each explanation references specific chapter sections and tables.

---

### Part 1 Solutions

<details>
<summary><strong>Q1 Solution — Click to reveal</strong></summary>

**(a)** No, the table is **not in 1NF**. The `courses` column contains a comma-separated list — multiple values in a single cell. The 1NF rule (Table 3 in the chapter) requires every cell to contain one atomic value with no repeating groups.

**(b)** Finding all students enrolled in Physics requires string parsing: `WHERE courses LIKE '%Physics%'`. This is fragile (it would also match "Astrophysics"), cannot use indexes efficiently, and makes aggregation (e.g., "how many students per course?") extremely difficult. The multi-valued column defeats the relational model's ability to enforce constraints and query efficiently.

**(c)** Decompose into two tables:

**Student:**

| student_id | student_name |
|---|---|
| S001 | Alice Chen |
| S002 | Bob Patel |
| S003 | Carol Liu |

**StudentCourse:**

| student_id | course_name |
|---|---|
| S001 | Math |
| S001 | Physics |
| S001 | Chemistry |
| S002 | Math |
| S002 | Biology |
| S003 | Physics |

**(d)** `Student` has PK `student_id`. `StudentCourse` has composite PK `(student_id, course_name)`. Now "find all students in Physics" is a simple: `SELECT student_id FROM StudentCourse WHERE course_name = 'Physics'`.

</details>

---

<details>
<summary><strong>Q2 Solution — Click to reveal</strong></summary>

**(a)** No, the table is **in 1NF but not 2NF**. The composite key is `(student_id, course_id)`, and there are non-key attributes that depend on only part of the key.

**(b)** Violating FDs:
- `student_id → student_name` — `student_name` depends on `student_id` alone, not on the full composite key.
- `course_id → course_title` — `course_title` depends on `course_id` alone, not on the full composite key.

These are **partial dependencies** — the 2NF violation.

**(c)** If student S001 is enrolled in three courses, the table has three rows for S001, each with the same `student_name`. The name is stored three times. If the student changes their name, three rows must be updated — and if only some are updated, the name drifts.

**(d)** Decompose into three tables:

- `Student(student_id PK, student_name)` — name depends on student alone
- `Course(course_id PK, course_title)` — title depends on course alone
- `Enrolment(student_id FK, course_id FK, grade)` — composite PK `(student_id, course_id)`; grade depends on the full key

**(e)** Yes, all three tables are in BCNF:
- `Student`: only determinant is `student_id` (candidate key). BCNF.
- `Course`: only determinant is `course_id` (candidate key). BCNF.
- `Enrolment`: only determinant is `{student_id, course_id}` (candidate key). No partial or transitive dependencies. BCNF.

</details>

---

<details>
<summary><strong>Q3 Solution — Click to reveal</strong></summary>

**(a)** No, it cannot have a 2NF violation. The primary key is `order_id` (a single attribute, not composite). The 2NF check only applies to composite keys — a single-column PK cannot have partial dependencies (there is no "part" of the key to depend on). So 2NF is satisfied trivially. The table is in 2NF.

**(b)** The transitive dependency: `order_id → customer_id → customer_name, customer_city`. The non-key attribute `customer_id` determines other non-key attributes (`customer_name`, `customer_city`). Equivalently: `customer_name` and `customer_city` depend on `customer_id`, which is not a candidate key of this table. This is a 3NF violation.

**(c)** If customer C001's city is updated in a separate Customer table but not in existing `OrderFlat` rows, the "revenue by city" report will show two entries: one for the old city name (from historical orders) and one for the new name. Revenue totals will be split across two rows, creating a discrepancy — exactly Failure 1 from the chapter.

**(d)** Decompose into:

- `Customer(customer_id PK, customer_name, customer_city)` — customer details stored once
- `OrderClean(order_id PK, customer_id FK, order_date, total)` — order references customer by FK

Now `customer_city` is stored once per customer, not once per order. Updating the city requires changing one row.

**(e)** The extra JOIN is a **feature**, not a bug. The JOIN exists because we removed the redundant copy of `customer_city` from the order table. The cost is one extra join per query that needs the city. The benefit is that the city is stored once and can never drift. This is the central trade-off of normalisation: query convenience vs data integrity. For OLTP workloads (where integrity is paramount), this trade-off is correct. For OLAP workloads (where query simplicity matters more), the star schema deliberately reverses it.

</details>

---

<details>
<summary><strong>Q4 Solution — Click to reveal</strong></summary>

**(a)** Two candidate keys:
- `{payment_id}` — determines all other attributes.
- `{order_id}` — because of the UNIQUE constraint, `order_id` also uniquely identifies each row and determines all other attributes.

**(b)** Both determinants are candidate keys:
- `payment_id → order_id, amount, payment_date, method` — `payment_id` is a candidate key. ✓
- `order_id → payment_id, amount, payment_date, method` — `order_id` is a candidate key. ✓

**(c)** The table is in **BCNF**. Every determinant is a candidate key. No violations.

**(d)** If instalments are allowed, the UNIQUE constraint on `order_id` is dropped. Now `order_id` no longer uniquely identifies a Payment row (multiple payments can reference the same order). `order_id` is no longer a candidate key. The only candidate key is `payment_id`.

The FD `order_id → amount, payment_date, method` no longer holds (different payments for the same order can have different amounts and dates). So no BCNF violation arises — the FD simply disappears. The table remains in BCNF with `payment_id` as the sole candidate key.

This shows how a business rule change (allowing instalments) changes the FD structure, which changes the candidate key analysis. The BCNF assessment depends on the business rules, not just the column structure.

</details>

---

### Part 2 Solutions

<details>
<summary><strong>Q5 Solution — Click to reveal</strong></summary>

**(a)** Compared to the Chapter 2 schema:
- **Added:** `Postcode(postcode, city)` — a new lookup table decomposed from `Customer` to eliminate the transitive dependency `postcode → city`.
- **Changed:** `Customer` — the `city` column is removed, replaced by a `postcode` FK referencing the new `Postcode` table.

This is the BCNF decomposition that prevents Failure 1 (truth drift on city names).

**(b)** **13 tables:** the 12 from Chapter 2 (Customer, Order_, Product, OrderLine, Payment, Shipping, Review, Supplier, SupplierProduct, Category, ProductCategory) plus the new Postcode lookup table. Note: if your Chapter 2 count was 11 (without a separate Department table), the count here is 12 + Postcode = 13. The exact count depends on whether Department is modelled separately or as an attribute of Category.

</details>

---

<details>
<summary><strong>Q6 Solution — Click to reveal</strong></summary>

**(a)** `Customer(customer_id, name, email, postcode)`:
- FDs: `customer_id → name, email, postcode` and `email → customer_id, name, postcode` (if email is unique).
- Candidate keys: `{customer_id}` and `{email}`.
- Every determinant is a candidate key. **BCNF.** ✓

**(b)** `OrderLine(order_id, product_id, quantity, unit_price)`:
- FDs: `{order_id, product_id} → quantity, unit_price`.
- Candidate key: `{order_id, product_id}`.
- No partial dependencies: `order_id` alone does not determine `quantity` (same order can have different quantities for different products). `product_id` alone does not determine `quantity` (same product can have different quantities in different orders).
- **BCNF.** ✓

**(c)** `Payment(payment_id, order_id, amount, payment_date, method)`:
- **Two candidate keys:** `{payment_id}` and `{order_id}` (because `order_id` is UNIQUE).
- Both FDs have candidate-key determinants. **BCNF.** ✓

**(d)** No violations found. All tables are in BCNF.

</details>

---

<details>
<summary><strong>Q7 Solution — Click to reveal</strong></summary>

**Option A — Snapshot:**

**(a)** Added `shipping_address VARCHAR(200)`, `shipping_city VARCHAR(50)`, `shipping_postcode VARCHAR(10)` to the `Shipping` table.

**(b)** Downshift type: **Snapshot** (history preservation).

**(c)** This is an **apparent** NF violation, not a real one. The shipping address records the address at time of shipment — a historical fact with its own lifecycle. The customer's current address is a different attribute. They may share values today but will diverge when the customer moves. Since they are different facts, storing both does not violate "one table, one fact."

**(d)** Safety mechanism: **Immutability.** The shipping address is set once when the order is shipped and never updated. Since it is never modified, it cannot drift.

**(e)** Example dbdiagram.io comment: `-- SNAPSHOT: shipping address at time of dispatch. Immutable. Not a copy of Customer.postcode — records historical delivery location.`

**Option B — Derived attribute:**

**(b)** Downshift type: **Derived attribute** (performance).

**(c)** This relaxes **3NF** — `order_total` is transitively determined by `order_id` through the `OrderLine` rows. It is a genuine controlled redundancy: the same information can be computed from `OrderLine`.

**(d)** Safety mechanism: **Transactional update.** A trigger (or application logic) recalculates `order_total` whenever an `OrderLine` row is inserted, updated, or deleted. Chapter 5 will implement this trigger.

**(e)** Example comment: `-- DERIVED: order_total = SUM(quantity * unit_price) from OrderLine. Maintained by trigger. Truth lives in OrderLine.`

</details>

---

### Part 3 Solutions

<details>
<summary><strong>Q8 Solution — Click to reveal</strong></summary>

**(b)** Grain: "One row represents **one line item in one order** — a specific product purchased in a specific order, on a specific date, by a specific customer."

**(c)** FD analysis for each dimension table:

- **ProductDim:** `product_key → product_name, category_name, department_name`. Also: `category_name → department_name` (each category belongs to one department). The determinant `category_name` is not a candidate key. **2NF but not 3NF.** ← predicted violation.
- **TimeDim:** `date_key → full_date, month, quarter, year`. Also: `full_date → month, quarter, year` and `{month, year} → quarter`. The determinant `full_date` is a candidate key (each date is unique, marked UNIQUE in the DDL). The determinant `{month, year}` may or may not be a candidate key depending on whether the time range spans multiple years. In practice, `date_key` and `full_date` are both candidate keys. **BCNF** (if `full_date` is unique) or **3NF** depending on the exact FDs. No violation to worry about in practice.
- **CustomerDim:** `customer_key → customer_name, city, region`. Also: `city → region` (each city is in one region). The determinant `city` is not a candidate key. **2NF but not 3NF.** ← predicted violation.

**(d)** The 3NF violations in `ProductDim` and `CustomerDim` are acceptable because:
1. **Read-only:** These dimension tables are never edited directly by transactional users.
2. **Derived from normalised truth:** They are rebuilt from the BCNF source (Product + Category + Department, Customer + Postcode).
3. **Controlled scope:** The violations are named and documented; only dimension tables contain redundancy.

</details>

---

<details>
<summary><strong>Q9 Solution — Click to reveal</strong></summary>

**(a)** The BCNF query needs **4 JOIN clauses**, touching **5 tables**: OrderLine, Product, Category, Department, and Order_. The join chain reconstructs the product hierarchy (OrderLine → Product → Category → Department) and adds the date (Order_).

**(c)** 4 JOINs, 5 tables.

**(d)** The result should show all 5 departments (Electronics, Clothing, Home & Garden, Books, Sports) with quarterly revenue figures for 2023 and 2024. The totals should reflect the random seed data — exact values depend on the generation, but each department should have non-zero revenue in most quarters.

</details>

---

<details>
<summary><strong>Q10 Solution — Click to reveal</strong></summary>

**(a)** The star schema query needs **2 JOIN clauses**, touching **3 tables**: OrderLineFact, ProductDim, and TimeDim. No join chain — each dimension connects directly to the fact table.

**(c)** 2 JOINs, 3 tables.

**(d)** The revenue totals should match the BCNF version (within minor rounding tolerance). Both queries aggregate the same underlying data — the star schema is derived from the BCNF source. If totals differ significantly, check whether the data generation script loaded both schemas consistently.

</details>

---

<details>
<summary><strong>Q11 Solution — Click to reveal</strong></summary>

**(a)** Expected comparison:

| Metric | BCNF query | Star schema query |
|---|---|---|
| Number of JOIN clauses | 4 | 2 |
| Number of tables touched | 5 | 3 |
| Execution time | (varies, typically higher) | (varies, typically lower) |
| Revenue totals match? | — | Yes |

**(b)** The star schema query has fewer JOINs because the dimension tables **pre-flatten the join chain**. `ProductDim` already contains `category_name` and `department_name` — the information that the BCNF version reconstructs by joining Product → Category → Department. The denormalisation in the dimension table eliminates intermediate tables from the query path.

**(c)** The **normalised `Department` table** is the source of truth. `ProductDim.department_name` is a derived copy — it was populated from the BCNF schema and should be regenerated whenever the source changes. The principle: "truth is normalised; reporting is derived." If someone queries `ProductDim` for the canonical department name, they are reading a snapshot, not the truth.

**(d)** If a department is renamed in the OLTP system:
1. Update the `Department` table in the BCNF schema (one row, one update).
2. **Regenerate** the star schema — rebuild `ProductDim` from the updated BCNF source.

If someone renamed it directly in `ProductDim` (without updating the BCNF source), the star schema would show the new name while the OLTP system shows the old one. This is the same truth-drift problem from Failure 1 — but between schemas rather than within one. The safety argument for the star schema depends on it being derived, never edited directly.

</details>

---

<details>
<summary><strong>Q12 Solution (Bonus) — Click to reveal</strong></summary>

**(a)** Run both queries with `WHERE department_name = 'Electronics'` (adjusting the table alias as needed). Record execution times.

**(b)** The performance gap may narrow slightly because the filter reduces the data volume for both queries. However, the structural difference (4 JOINs vs 2 JOINs) remains, so the star schema version should still be faster. On small datasets (~50K rows), the difference may be minimal; on production-scale data (millions of rows), the gap widens significantly because each JOIN multiplies the optimiser's work.

</details>